# Trabajo final entregable: biofertilizantes en gramineas

Este notebook integra el **codigo entregable** del trabajo final y cubre de forma articulada la **Seccion 2** (flujo de trabajo, gramatica de los graficos y reproducibilidad) y la **Seccion 3** (EDA, visualizacion biologica avanzada e interpretacion de decisiones).

El objetivo no es solo generar figuras, sino dejar trazable el paso desde el dato crudo hasta visualizaciones defendibles desde la rubrica de evaluacion del curso.


## Cobertura de la rubrica

Este notebook esta organizado para responder explicitamente a tres bloques:

- **Flujo de trabajo y datos**: carga, filtros, limpieza y construccion de tablas tidy.
- **Mapeo gramatical**: cada figura explicita datos, transformaciones, geometrias y escalas.
- **Seccion 3**: EDA univariado y bivariado, visualizacion biologica avanzada y decisiones de diseño.

Ademas, todas las figuras evitan soluciones 3D y priorizan comparaciones por posicion, escalas compartidas, visibilidad del dato crudo y paletas perceptualmente honestas.


In [1]:
from pathlib import Path
import re
import unicodedata
import warnings

import altair as alt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.cluster.hierarchy import leaves_list, linkage
from scipy.spatial.distance import pdist

warnings.filterwarnings("ignore")
alt.data_transformers.disable_max_rows()

CROP_PALETTE = {
    "Maiz": "#66c2a5",
    "Sorgo/Maicillo": "#fc8d62",
    "Arroz": "#8da0cb"
}
EDA_COLORS = {
    "crops": "#7c8da6",
    "biofert": "#9c7c68",
    "hist": "#6b9080",
    "plag_theme": "#5f7f6f",
    "dev_theme": "#b07d62"
}

ROOT = Path.cwd().resolve()
if not (ROOT / "base_datos").exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / "base_datos" / "biofertilizantes (2).xlsx"

print("Repositorio:", ROOT)
print("Excel fuente:", DATA_PATH)
print("Pandas:", pd.__version__)
print("Altair:", alt.__version__)


Repositorio: C:\Users\ZAIRA\Desktop\visualizacion_proyecto
Excel fuente: C:\Users\ZAIRA\Desktop\visualizacion_proyecto\base_datos\biofertilizantes (2).xlsx
Pandas: 2.2.3
Altair: 5.5.0


## 1. Flujo de trabajo y datos

La hoja primaria del pipeline es `Respuestas de formulario 1`, porque conserva el nivel de observacion por agricultor. Las demas hojas del Excel se consideran auxiliares, pero no sustituyen al dato crudo.

Los filtros iniciales son parte del metodo, no una limpieza accidental:

- se elimina una fila completamente vacia;
- se elimina una fila espuria con `cultivo = 11`;
- se excluye una respuesta fuera del dominio biologico del estudio (`citricos, naranja, papaya y tomate`).


In [2]:
raw = pd.read_excel(DATA_PATH, sheet_name="Respuestas de formulario 1")

mask_not_blank = raw.iloc[:, 2].notna()
mask_not_bad = raw.iloc[:, 2].astype(str).str.strip() != "11"
mask_gramineas = ~raw.iloc[:, 2].astype(str).str.contains("citricos", case=False, na=False)

survey = raw.loc[mask_not_blank & mask_not_bad & mask_gramineas].copy().reset_index(drop=True)
survey.insert(0, "id_encuesta", range(1, len(survey) + 1))

print("Dimensiones crudas:", raw.shape)
print("Dimensiones analiticas:", survey.shape)
print("Filas excluidas:", len(raw) - len(survey))
survey.iloc[:4, :8]


Dimensiones crudas: (28, 25)
Dimensiones analiticas: (25, 26)
Filas excluidas: 3


,id_encuesta,Marca temporal,Nombre comercial del biofertilizante que utiliza,Tipo de cultivo en el que utiliza biofertilizantes,Número de manzanas que cultiva,"Si conoce los principales compuestos del biofertilizante, escríbalos a continuación:",¿Cuál fue la producción en quintales que obtuvo en su última cosecha por cada manzana de cultivo?,"¿Ha notado algún cambio en el nivel de desarrollo de sus plantas después de aplicar el biofertilizante? En caso afirmativo, ¿cómo describiría estos cambios?"
0,1,2023-05-17 10:41:42.651,nitroesten nitrogenadoo granulado quintal,Maíz,0.5 mz,nitrogeno,60 en media manzana,"mejor color, vigorosa"
1,2,2023-05-17 15:06:11.121,Biopro,Maíz,2,no,80 qq por manzana,"Si, es mas vigoroza"
2,3,2023-05-17 15:11:23.476,BIOPRO,Maíz,5,NO,78 qq por manzana,"Si,"
3,4,2023-05-17 15:15:57.742,biopro,Maíz,1/2 manzana,no,37 qq,"si, se ve mas saludable"


In [3]:
def normalize_text(value):
    if pd.isna(value):
        return ""
    text = str(value).strip().lower()
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"\s+", " ", text)


def clean_biofert(value):
    text = normalize_text(value)
    if "biopro" in text:
        return "Biopro"
    if "bio amigo" in text or "bioamigo" in text:
        return "Bioamigo"
    if "organosato" in text or "orgosanto" in text:
        return "Organosato"
    if "crop" in text:
        return "Crop Plus"
    if "albamin" in text:
        return "Albamin"
    if "nitro" in text:
        return "Nitroesten"
    if "bocachi" in text:
        return "Bocachi"
    if any(token in text for token in ["micorriza", "polvo de roca", "gallinaza", "rastrojo", "vacaza"]):
        return "Mezclas artesanales"
    return "Otro"


def clean_crop_reported(value):
    text = normalize_text(value)
    if text == "maiz":
        return "Maiz"
    if text == "sorgo/maicillo":
        return "Sorgo/Maicillo"
    if text == "arroz":
        return "Arroz"
    if "maiz" in text and "sorgo" in text and "frijol" in text:
        return "Maiz + Sorgo + Frijol"
    if "maiz" in text and "sorgo" in text and "cana" in text:
        return "Maiz + Sorgo + Cana"
    if "maiz" in text and "sorgo" in text:
        return "Maiz + Sorgo"
    return "Otro"


def clean_crop_principal(value):
    text = normalize_text(value)
    if "maiz" in text:
        return "Maiz"
    if "sorgo" in text or "maicillo" in text:
        return "Sorgo/Maicillo"
    if "arroz" in text:
        return "Arroz"
    if "cana" in text:
        return "Cana de azucar"
    return "Otro"


def parse_production(value):
    text = normalize_text(value)
    if text == "" or "forraje" in text or "no aplica" in text:
        return np.nan
    numbers = re.findall(r"\d+\.?\d*", text)
    return float(numbers[0]) if numbers else np.nan


def is_out_of_domain_pesticide_response(value):
    text = normalize_text(value)
    if text == "":
        return False
    fertilizer_formula = re.search(r"\b\d{1,2}\s*-\s*\d{1,2}\s*-\s*\d{1,2}\b", text)
    fertilization_schedule = "siembra" in text and any(token in text for token in ["primera", "segunda", "tercera"])
    return bool(fertilizer_formula or fertilization_schedule)


def parse_pesticide_frequency(value):
    text = normalize_text(value)
    if text == "" or "depende" in text:
        return np.nan
    if is_out_of_domain_pesticide_response(text):
        return np.nan
    if "cada" in text and "dia" in text and "vez" not in text and "bombada" not in text and "abonada" not in text:
        return np.nan
    range_match = re.search(r"(\d+)\s*-\s*(\d+)(?!\s*-\s*\d+)", text)
    if range_match:
        return (float(range_match.group(1)) + float(range_match.group(2))) / 2
    count_like = any(token in text for token in ["vez", "veces", "bombada", "bombadas", "abonada", "abonadas", "sola vez"])
    numbers = re.findall(r"\d+\.?\d*", text)
    if count_like and numbers:
        return float(numbers[0])
    return np.nan


def classify_plague_change(value):
    text = normalize_text(value)
    if any(token in text for token in ["disminu", "menos", "bajo", "bajado", "dismiuyo"]):
        return "Disminucion"
    if text in {"no", "lo mismo"} or any(token in text for token in ["igual", "similar", "mismo"]):
        return "Sin cambio o mixto"
    return "Ambiguo"


def classify_plague_text_theme(value):
    text = normalize_text(value)
    if any(token in text for token in ["menos", "disminu", "bajo", "poco"]):
        return "Menor uso o incidencia"
    if any(token in text for token in ["resiste", "proteg", "protegida", "virus", "vector"]):
        return "Mayor resistencia o proteccion"
    if any(token in text for token in ["igual", "mismo", "similar", "no"]):
        return "Sin cambio claro"
    return "Otro o ambiguo"


def classify_development_theme(value):
    text = normalize_text(value)
    if any(token in text for token in ["vigor", "vigorosa", "vigoroso"]):
        return "Vigor"
    if any(token in text for token in ["raiz", "raices"]):
        return "Raices"
    if any(token in text for token in ["verde", "verdor", "color"]):
        return "Verdor o color"
    if any(token in text for token in ["germin", "nace", "rebrote"]):
        return "Germinacion o rebrote"
    if any(token in text for token in ["llenado", "grano", "semilla"]):
        return "Llenado de grano o semilla"
    if any(token in text for token in ["salud", "saludable", "mejor"]):
        return "Salud general"
    return "Otro o ambiguo"


survey_clean = pd.DataFrame({
    "id_encuesta": survey["id_encuesta"],
    "biofert_std": survey.iloc[:, 2].apply(clean_biofert),
    "cultivo_reportado_std": survey.iloc[:, 3].apply(clean_crop_reported),
    "cultivo_principal": survey.iloc[:, 3].apply(clean_crop_principal),
    "manzanas_cultivadas": survey.iloc[:, 4],
    "produccion_qq_mz": survey.iloc[:, 6].apply(parse_production),
    "texto_desarrollo": survey.iloc[:, 7],
    "respuesta_freq_plaguicidas": survey.iloc[:, 8],
    "freq_plaguicidas_fuera_dominio": survey.iloc[:, 8].apply(is_out_of_domain_pesticide_response),
    "freq_plaguicidas_ciclo": survey.iloc[:, 8].apply(parse_pesticide_frequency),
    "texto_plaguicidas": survey.iloc[:, 9],
    "cambio_plagas_std": survey.iloc[:, 10].apply(classify_plague_change),
    "tema_plaguicidas": survey.iloc[:, 9].apply(classify_plague_text_theme),
    "tema_desarrollo": survey.iloc[:, 7].apply(classify_development_theme)
})

LIKERT_MAP = {
    "Muy en desacuerdo": 1,
    "Algo en desacuerdo": 2,
    "Ni de acuerdo ni desacuerdo": 3,
    "Parcialmente de acuerdo": 4,
    "Totalmente de acuerdo": 5
}

ITEM_LABELS = {
    survey.columns[12]: "Cambios negativos",
    survey.columns[13]: "Cultivo mas saludable",
    survey.columns[14]: "Resiste malas condiciones",
    survey.columns[15]: "Mas vigoroso que vecinos",
    survey.columns[16]: "Mas vigoroso desde uso",
    survey.columns[17]: "Satisfaccion general",
    survey.columns[18]: "Precio-calidad",
    survey.columns[19]: "Recomendaria",
    survey.columns[20]: "Mejora calidad cosecha",
    survey.columns[21]: "Cambio en ganancias",
    survey.columns[22]: "Buen rendimiento",
    survey.columns[23]: "Cumple necesidades",
    survey.columns[24]: "Seguira usando"
}

likert_wide = survey_clean[["id_encuesta", "biofert_std", "cultivo_principal"]].copy()
for raw_col, short_col in ITEM_LABELS.items():
    likert_wide[short_col] = survey[raw_col].map(LIKERT_MAP)

likert_long = likert_wide.melt(
    id_vars=["id_encuesta", "biofert_std", "cultivo_principal"],
    var_name="item",
    value_name="score"
).dropna()

print("N respuestas validas:", len(survey_clean))
print("Produccion disponible:", survey_clean["produccion_qq_mz"].notna().sum())
print("Frecuencia de plaguicidas disponible:", survey_clean["freq_plaguicidas_ciclo"].notna().sum())
print("Frecuencia de plaguicidas fuera de dominio:", int(survey_clean["freq_plaguicidas_fuera_dominio"].sum()))
print("\nCasos fuera de dominio en frecuencia de plaguicidas:")
print(survey_clean.loc[survey_clean["freq_plaguicidas_fuera_dominio"], ["id_encuesta", "biofert_std", "respuesta_freq_plaguicidas"]].to_string(index=False))
print("\nBiofertilizantes normalizados:")
print(survey_clean["biofert_std"].value_counts().to_string())
print("\nCultivo principal:")
print(survey_clean["cultivo_principal"].value_counts().to_string())
survey_clean.head()


N respuestas validas: 25
Produccion disponible: 24
Frecuencia de plaguicidas disponible: 17
Frecuencia de plaguicidas fuera de dominio: 1

Casos fuera de dominio en frecuencia de plaguicidas:
 id_encuesta biofert_std                                                                                          respuesta_freq_plaguicidas
           1  Nitroesten 3 abonadas y quintal por aonada, 16-20-0 al momento de siembra. a los 15 dias primera,  segunda 25, tercera 35 dias

Biofertilizantes normalizados:
biofert_std
Biopro                 9
Bioamigo               6
Mezclas artesanales    3
Organosato             2
Crop Plus              2
Nitroesten             1
Bocachi                1
Albamin                1

Cultivo principal:
cultivo_principal
Maiz              19
Sorgo/Maicillo     3
Arroz              3


,id_encuesta,biofert_std,cultivo_reportado_std,cultivo_principal,manzanas_cultivadas,produccion_qq_mz,texto_desarrollo,respuesta_freq_plaguicidas,freq_plaguicidas_fuera_dominio,freq_plaguicidas_ciclo,texto_plaguicidas,cambio_plagas_std,tema_plaguicidas,tema_desarrollo
0,1,Nitroesten,Maiz,Maiz,0.5 mz,60.0,"mejor color, vigorosa","3 abonadas y quintal por aonada, 16-20-0 al mo...",True,NaN,etsal liquido 12cc por bombada 17 litros,Disminucion,Otro o ambiguo,Vigor
1,2,Biopro,Maiz,Maiz,2,80.0,"Si, es mas vigoroza",2 bombadas por manzana,False,2.0,"Si, es menor la insidencia despues de usarla",Disminucion,Sin cambio claro,Vigor
2,3,Biopro,Maiz,Maiz,5,78.0,"Si,",depende de la incidencia de plagas,False,NaN,"Si, ha disminuido",Disminucion,Menor uso o incidencia,Otro o ambiguo
3,4,Biopro,Maiz,Maiz,1/2 manzana,37.0,"si, se ve mas saludable",depende de como vaya la plaga,False,NaN,si se observan mucho menos,Disminucion,Menor uso o incidencia,Salud general
4,5,Biopro,Maiz,Maiz,8,80.0,"SI, SE VE MAS VERDE","DEPENDE DE CUANTA PLAGA TENGA, ENTRE 2 BOMBADAS",False,NaN,"SI, LA PLANTA RESISTE MAS LA PLAGA",Disminucion,Mayor resistencia o proteccion,Verdor o color


In [4]:
missingness = (
    survey_clean.isna()
    .mean()
    .sort_values(ascending=False)
    .rename("proporcion_na")
    .reset_index()
    .rename(columns={"index": "variable"})
)

print("Variables con mayor proporción de faltantes:")
print(missingness.to_string(index=False))

print("\nMedia Likert por item:")
print(likert_long.groupby("item")["score"].mean().sort_values(ascending=False).round(2).to_string())


Variables con mayor proporción de faltantes:
                      variable  proporcion_na
        freq_plaguicidas_ciclo           0.32
              produccion_qq_mz           0.04
                   biofert_std           0.00
         cultivo_reportado_std           0.00
             cultivo_principal           0.00
                   id_encuesta           0.00
           manzanas_cultivadas           0.00
              texto_desarrollo           0.00
    respuesta_freq_plaguicidas           0.00
freq_plaguicidas_fuera_dominio           0.00
             texto_plaguicidas           0.00
             cambio_plagas_std           0.00
              tema_plaguicidas           0.00
               tema_desarrollo           0.00

Media Likert por item:
item
Mas vigoroso desde uso       4.40
Recomendaria                 4.40
Satisfaccion general         4.36
Mejora calidad cosecha       4.24
Cultivo mas saludable        4.24
Seguira usando               4.20
Cumple necesidades           4.0

## 2. EDA: auditoria inicial del dataset

La Seccion 3 exige EDA univariado y bivariado. Antes de pasar a visualizaciones mas interpretativas, conviene auditar la composicion muestral, la distribucion de biofertilizantes y la disponibilidad de datos cuantitativos.


In [5]:
crop_counts = survey_clean["cultivo_principal"].value_counts().rename_axis("cultivo").reset_index(name="n")
biofert_counts = survey_clean["biofert_std"].value_counts().rename_axis("biofert").reset_index(name="n")
prod_hist_df = survey_clean.dropna(subset=["produccion_qq_mz"]).copy()

chart_crops = alt.Chart(crop_counts).mark_bar(color=EDA_COLORS["crops"], cornerRadiusEnd=4).encode(
    x=alt.X("n:Q", title="Encuestados"),
    y=alt.Y("cultivo:N", sort="-x", title="Cultivo principal"),
    tooltip=["cultivo:N", "n:Q"]
).properties(width=260, height=160, title="Composicion por cultivo")

chart_biofert = alt.Chart(biofert_counts).mark_bar(color=EDA_COLORS["biofert"], cornerRadiusEnd=4).encode(
    x=alt.X("n:Q", title="Encuestados"),
    y=alt.Y("biofert:N", sort="-x", title="Biofertilizante"),
    tooltip=["biofert:N", "n:Q"]
).properties(width=300, height=180, title="Composicion por biofertilizante")

chart_hist = alt.Chart(prod_hist_df).mark_bar(color=EDA_COLORS["hist"], opacity=0.85).encode(
    x=alt.X("produccion_qq_mz:Q", bin=alt.Bin(maxbins=10), title="Produccion (qq por manzana)"),
    y=alt.Y("count():Q", title="Frecuencia"),
    tooltip=[alt.Tooltip("count():Q", title="N")]
).properties(width=620, height=180, title="Distribucion global de la produccion declarada")

alt.vconcat(
    alt.hconcat(chart_crops, chart_biofert),
    chart_hist,
    spacing=18
).configure_view(strokeWidth=0).configure_axis(gridColor="#e6e6e6", domainColor="#bdbdbd")


alt.VConcatChart(...)

### Interpretacion del EDA inicial

La muestra no esta balanceada: el maiz domina claramente y varios biofertilizantes aparecen con muy pocos casos. Esta observacion es importante para la honestidad visual, porque obliga a evitar comparaciones basadas solo en promedios o conteos absolutos no normalizados. Ademas, la auditoria inicial muestra que no todos los textos semiestructurados pueden convertirse de forma literal a numeros: una respuesta de Nitroesten en la variable de frecuencia de plaguicidas describe un calendario de fertilizacion (`16-20-0`, `siembra`, `primera`, `segunda`, `tercera`) y por eso se trata como dato fuera de dominio, no como un valor extremo valido. Tambien justifica que algunas figuras se centren en subgrupos comparables y que otras incorporen `n` de forma explicita.


In [6]:
df_maiz = survey_clean[(survey_clean["cultivo_principal"] == "Maiz") & survey_clean["produccion_qq_mz"].notna()].copy()
order_prod = (
    df_maiz.groupby("biofert_std")["produccion_qq_mz"]
    .median()
    .sort_values(ascending=False)
    .index
    .tolist()
)

points = alt.Chart(df_maiz).mark_circle(size=90, opacity=0.75).encode(
    x=alt.X("biofert_std:N", sort=order_prod, title="Biofertilizante"),
    y=alt.Y("produccion_qq_mz:Q", title="Produccion declarada (qq por manzana)"),
    color=alt.value("#1f77b4"),
    tooltip=["id_encuesta:Q", "biofert_std:N", alt.Tooltip("produccion_qq_mz:Q", format=".1f")]
)

box = alt.Chart(df_maiz).mark_boxplot(size=28, opacity=0.25, outliers=False, color="#333333").encode(
    x=alt.X("biofert_std:N", sort=order_prod),
    y=alt.Y("produccion_qq_mz:Q")
)

rule = alt.Chart(pd.DataFrame({"y": [50]})).mark_rule(color="#b36b00", strokeDash=[5, 3]).encode(y="y:Q")

(points + box + rule).properties(
    width=620,
    height=320,
    title="Produccion por biofertilizante en casos comparables de maiz"
).configure_view(strokeWidth=0)


alt.LayerChart(...)

### Decisiones de diseño: produccion

Se eligio un `strip plot` superpuesto con `boxplot` porque el dataset es pequeno y cada agricultor cuenta. Esta figura evita el error de mostrar solo promedios y permite ver dispersion, valores extremos y consistencia interna por biofertilizante. La linea de referencia en `50 qq/manzana` contextualiza la lectura para maiz y evita que la magnitud de las diferencias quede sin marco interpretativo.


In [7]:
df_plag = (
    survey_clean.dropna(subset=["freq_plaguicidas_ciclo"])
    .groupby("biofert_std")
    .agg(media=("freq_plaguicidas_ciclo", "mean"), n=("freq_plaguicidas_ciclo", "size"))
    .reset_index()
    .sort_values("media")
)
df_plag["cero"] = 0
df_plag["label"] = df_plag["biofert_std"] + " (n=" + df_plag["n"].astype(str) + ")"

segments = alt.Chart(df_plag).mark_rule(color="#c7c7c7").encode(
    y=alt.Y("label:N", sort=list(df_plag["label"]), title=None),
    x=alt.X("cero:Q", title="Aplicaciones de plaguicidas por ciclo (media)"),
    x2="media:Q"
)

dots = alt.Chart(df_plag).mark_circle(size=120, color="#e45756").encode(
    y=alt.Y("label:N", sort=list(df_plag["label"]), title=None),
    x="media:Q",
    tooltip=["biofert_std:N", alt.Tooltip("media:Q", format=".2f"), "n:Q"]
)

labels = alt.Chart(df_plag).mark_text(align="left", dx=8).encode(
    y=alt.Y("label:N", sort=list(df_plag["label"]), title=None),
    x="media:Q",
    text=alt.Text("media:Q", format=".2f")
)

df_scatter = survey_clean.dropna(subset=["produccion_qq_mz", "freq_plaguicidas_ciclo"]).copy()

scatter = alt.Chart(df_scatter).mark_circle(size=100, opacity=0.8).encode(
    x=alt.X("freq_plaguicidas_ciclo:Q", title="Aplicaciones de plaguicidas por ciclo"),
    y=alt.Y("produccion_qq_mz:Q", title="Produccion (qq por manzana)"),
    color=alt.Color("cultivo_principal:N", title="Cultivo principal"),
    tooltip=["id_encuesta:Q", "biofert_std:N", "cultivo_principal:N", alt.Tooltip("freq_plaguicidas_ciclo:Q", format=".2f"), alt.Tooltip("produccion_qq_mz:Q", format=".1f")]
)

reg = alt.Chart(df_scatter).transform_regression(
    "freq_plaguicidas_ciclo", "produccion_qq_mz", method="linear"
).mark_line(color="#444444", strokeDash=[4, 4]).encode(
    x="freq_plaguicidas_ciclo:Q",
    y="produccion_qq_mz:Q"
)

alt.hconcat(
    (segments + dots + labels).properties(width=330, height=220, title="Organosato y Biopro muestran las medias mas bajas de plaguicidas"),
    (scatter + reg).properties(width=330, height=220, title="La relacion produccion-plaguicidas no es lineal ni univoca"),
    spacing=18
).configure_view(strokeWidth=0)


alt.HConcatChart(...)

### Decisiones de diseño: plaguicidas y relacion bivariada

La frecuencia media de plaguicidas se representa con un `Cleveland dot plot` porque la comparacion por posicion en eje comun es mas precisa que las barras o los pasteles. Sin embargo, esta figura solo es interpretable despues de una limpieza semantica estricta: la respuesta de Nitroesten que contenia `16-20-0` y un calendario de abonado se reclasifica como fuera de dominio y pasa a `NA`, porque describe fertilizacion y no aplicaciones de plaguicidas. Al eliminar ese falso extremo, el eje horizontal deja de comprimirse artificialmente y la comparacion entre Organosato, Biopro, Bioamigo y las mezclas artesanales se vuelve legible. Ademas, se explicita el tamaño muestral por biofertilizante para evitar lecturas infladas en categorias con pocos casos. El diagrama de dispersion complementa la lectura mostrando que, aun sin ese punto de leverage, la relacion entre produccion y plaguicidas no es lineal ni univoca, lo cual refuerza una interpretacion prudente del patron biologico.


In [8]:
likert_text = survey[["id_encuesta"] + list(ITEM_LABELS.keys())].melt(
    id_vars="id_encuesta",
    var_name="item_raw",
    value_name="respuesta"
).dropna()
likert_text["item"] = likert_text["item_raw"].map(ITEM_LABELS)

likert_order = [
    "Muy en desacuerdo",
    "Algo en desacuerdo",
    "Ni de acuerdo ni desacuerdo",
    "Parcialmente de acuerdo",
    "Totalmente de acuerdo"
]

counts = (
    likert_text.groupby(["item", "respuesta"])
    .size()
    .reset_index(name="n")
)
counts["pct"] = counts.groupby("item")["n"].transform(lambda s: 100 * s / s.sum())

records = []
for row in counts.itertuples(index=False):
    if row.respuesta == "Ni de acuerdo ni desacuerdo":
        records.append({"item": row.item, "respuesta": row.respuesta, "value": -(row.pct / 2)})
        records.append({"item": row.item, "respuesta": row.respuesta, "value": row.pct / 2})
    elif row.respuesta in {"Muy en desacuerdo", "Algo en desacuerdo"}:
        records.append({"item": row.item, "respuesta": row.respuesta, "value": -row.pct})
    else:
        records.append({"item": row.item, "respuesta": row.respuesta, "value": row.pct})

likert_plot = pd.DataFrame(records)
item_order = (
    likert_long.groupby("item")["score"]
    .mean()
    .sort_values(ascending=False)
    .index
    .tolist()
)

alt.Chart(likert_plot).mark_bar().encode(
    y=alt.Y("item:N", sort=item_order, title=None),
    x=alt.X("value:Q", stack="zero", title="Balance de respuestas (%): desacuerdo <- 0 -> acuerdo", axis=alt.Axis(values=[-100, -50, 0, 50, 100])),
    color=alt.Color(
        "respuesta:N",
        sort=likert_order,
        scale=alt.Scale(
            domain=likert_order,
            range=["#b2182b", "#ef8a62", "#d9d9d9", "#67a9cf", "#2166ac"]
        ),
        title="Respuesta"
    ),
    tooltip=["item:N", "respuesta:N", alt.Tooltip("value:Q", format=".2f")]
).properties(
    width=700,
    height=340,
    title="Barras divergentes para las 13 preguntas Likert"
).configure_view(strokeWidth=0)


alt.Chart(...)

In [9]:
plag_theme_counts = survey_clean["tema_plaguicidas"].value_counts().rename_axis("tema").reset_index(name="n")
dev_theme_counts = survey_clean["tema_desarrollo"].value_counts().rename_axis("tema").reset_index(name="n")

chart_plag_theme = alt.Chart(plag_theme_counts).mark_bar(color=EDA_COLORS["plag_theme"], cornerRadiusEnd=4).encode(
    x=alt.X("n:Q", title="Respuestas"),
    y=alt.Y("tema:N", sort="-x", title=None),
    tooltip=["tema:N", "n:Q"]
).properties(width=320, height=180, title="Temas sobre cambios en plaguicidas")

chart_dev_theme = alt.Chart(dev_theme_counts).mark_bar(color=EDA_COLORS["dev_theme"], cornerRadiusEnd=4).encode(
    x=alt.X("n:Q", title="Respuestas"),
    y=alt.Y("tema:N", sort="-x", title=None),
    tooltip=["tema:N", "n:Q"]
).properties(width=320, height=180, title="Temas sobre desarrollo del cultivo")

display(
    alt.hconcat(chart_plag_theme, chart_dev_theme, spacing=18)
    .configure_view(strokeWidth=0)
    .configure_axis(gridColor="#e6e6e6", domainColor="#bdbdbd")
)


alt.HConcatChart(...)

### Decisiones de diseno: Likert y respuestas abiertas

Las preguntas Likert se representan con barras divergentes porque la escala tiene direccion semantica y debe leerse alrededor de un punto neutro. Esta solucion conserva la estructura de cada item, a diferencia de los agregados `Bajo`, `Medio` y `Alto` del manuscrito original. Para las respuestas abiertas, la salida principal se mantiene en una codificacion tematica reproducible con barras ordenadas y eje comun, porque es la opcion mas analitica, comparable y defendible para el entregable. Esta decision evita volver a recursos menos precisos y mantiene la lectura centrada en categorias biologicamente interpretables.


In [10]:
MIN_DISTRIBUTION_N = 5

df_sm = survey_clean[
    survey_clean["produccion_qq_mz"].notna() &
    survey_clean["cultivo_principal"].isin(["Maiz", "Sorgo/Maicillo", "Arroz"])
].copy()

df_sm = df_sm.sort_values(["cultivo_principal", "produccion_qq_mz"]).reset_index(drop=True)
df_sm["rank_in_crop"] = df_sm.groupby("cultivo_principal").cumcount()
df_sm["n_crop"] = df_sm.groupby("cultivo_principal")["id_encuesta"].transform("size")
max_n_crop = int(df_sm["n_crop"].max())
df_sm["y_pos"] = df_sm["rank_in_crop"] - (df_sm["n_crop"] - 1) / 2
df_sm["mostrar_resumen"] = df_sm["n_crop"] >= MIN_DISTRIBUTION_N
df_sm["valor_label"] = df_sm["produccion_qq_mz"].map(lambda x: f"{x:.0f}")

base_sm = alt.Chart(df_sm).encode(
    x=alt.X("produccion_qq_mz:Q", title="Produccion (qq por manzana)", scale=alt.Scale(domain=[0, 220]), axis=alt.Axis(values=list(range(0, 221, 20)))),
    color=alt.Color("cultivo_principal:N", scale=alt.Scale(domain=list(CROP_PALETTE.keys()), range=list(CROP_PALETTE.values())), legend=None)
)

points_sm = base_sm.mark_circle(size=95, opacity=0.88).encode(
    y=alt.Y("y_pos:Q", title=None, axis=None, scale=alt.Scale(domain=[-(max_n_crop - 1) / 2 - 0.75, (max_n_crop - 1) / 2 + 0.75])),
    tooltip=["id_encuesta:Q", "cultivo_principal:N", "n_crop:Q", alt.Tooltip("produccion_qq_mz:Q", format=".1f", title="Produccion")]
)

value_labels_sm = alt.Chart(df_sm).transform_filter(
    alt.datum.n_crop < MIN_DISTRIBUTION_N
).mark_text(
    align="left",
    dx=8,
    fontSize=10,
    color="#444444"
).encode(
    x="produccion_qq_mz:Q",
    y="y_pos:Q",
    text="valor_label:N"
)

median_sm = alt.Chart(df_sm).transform_filter(
    alt.datum.mostrar_resumen
).mark_rule(color="#222222", strokeWidth=2).transform_joinaggregate(
    mediana="median(produccion_qq_mz)",
    groupby=["cultivo_principal"]
).encode(
    x="mediana:Q",
    y=alt.value(0)
)

n_sm = alt.Chart(df_sm).transform_aggregate(
    n="count()",
    groupby=["cultivo_principal"]
).transform_calculate(
    label='"n=" + datum.n + (datum.n < 5 ? " | valores individuales; sin mediana" : " | mediana mostrada")'
).mark_text(dy=-34, align="left", color="#444444", fontSize=11).encode(
    x=alt.value(8),
    y=alt.value(0),
    text="label:N"
)

small_multiples = alt.layer(points_sm, value_labels_sm, median_sm, n_sm, data=df_sm).facet(
    facet=alt.Facet("cultivo_principal:N", sort=["Maiz", "Sorgo/Maicillo", "Arroz"], title="Cultivo principal"),
    columns=3
).properties(
    title="Dot plots facetados por cultivo: se omite la mediana cuando n < 5"
).configure_view(strokeWidth=0).configure_axis(gridColor="#e6e6e6", domainColor="#bdbdbd")

small_multiples


alt.FacetChart(...)

In [11]:
heatmap_matrix = likert_wide.groupby("biofert_std")[list(ITEM_LABELS.values())].mean()
row_order = list(heatmap_matrix.index[leaves_list(linkage(pdist(heatmap_matrix.values), method="ward"))])
col_order = list(heatmap_matrix.columns[leaves_list(linkage(pdist(heatmap_matrix.values.T), method="ward"))])

heatmap_long = heatmap_matrix.reset_index().melt(id_vars="biofert_std", var_name="item", value_name="media")

heat = alt.Chart(heatmap_long).mark_rect(stroke="white", strokeWidth=1).encode(
    x=alt.X("item:N", sort=col_order, title=None),
    y=alt.Y("biofert_std:N", sort=row_order, title=None),
    color=alt.Color("media:Q", scale=alt.Scale(scheme="viridis", domain=[1, 5]), title="Media Likert"),
    tooltip=["biofert_std:N", "item:N", alt.Tooltip("media:Q", format=".2f")]
)

text = alt.Chart(heatmap_long).mark_text(fontSize=9).encode(
    x=alt.X("item:N", sort=col_order),
    y=alt.Y("biofert_std:N", sort=row_order),
    text=alt.Text("media:Q", format=".1f"),
    color=alt.condition(alt.datum.media >= 3.5, alt.value("white"), alt.value("#222222"))
)

(heat + text).properties(
    width=700,
    height=240,
    title="Heatmap de satisfaccion media por biofertilizante, ordenado por clustering jerarquico"
).configure_view(strokeWidth=0)


alt.LayerChart(...)

## Cierre de la Seccion 3: interpretacion de decisiones

Las figuras avanzadas del notebook responden directamente a la rubrica. En esta version, los `small multiples` dejan de ser histogramas facetados y pasan a `dot plots` con escala horizontal comun. Sin embargo, el notebook evita tratar como distribuciones a paneles con muestra insuficiente: la mediana solo se dibuja cuando `n >= 5`, mientras que en `Sorgo/Maicillo` y `Arroz` se muestran valores individuales anotados y una nota explicita de `n`. Asi, el panel de maiz funciona como comparacion principal y los otros cultivos quedan como contexto descriptivo, no como evidencia visualmente equiparable. Del mismo modo, la comparacion de plaguicidas deja de apoyarse en una extraccion numerica ingenua: el caso de Nitroesten se documenta como respuesta fuera de dominio y se excluye del calculo de `freq_plaguicidas_ciclo`, lo que evita comprimir artificialmente el Cleveland dot plot y distorsionar la recta del scatter por un falso valor extremo. El `heatmap` ordenado por clustering jerarquico funciona como una analogia biologica pertinente para un dataset no omico: no busca imitar artificialmente una matriz de expresion genetica, sino reutilizar una estrategia de visualizacion avanzada para detectar perfiles similares de percepcion y satisfaccion entre biofertilizantes.

Desde el punto de vista del dise?o, se tomaron cuatro decisiones transversales:

- **No usar 3D**: por precision perceptiva y por consistencia con los lineamientos del curso.
- **Priorizar posicion sobre area o volumen**: por eso se favorecen dot plots, barras ordenadas, strip plots y heatmaps.
- **Mostrar el tama?o muestral o el dato crudo cuando es importante**: especialmente en comparaciones entre biofertilizantes con pocos casos.
- **Mantener interpretaciones prudentes**: las figuras sugieren patrones, pero no fuerzan conclusiones causales que el tama?o de muestra no permite sostener.
